# Kaggle distillation with Unsloth

Run Qwen3-8B in 4-bit on one GPU with 100 fixed evaluation cases and ten rounds of 100 new pairs each (60 general and 40 adaptive). Enable Kaggle Internet and GPU access. Attach the matching `colab_distillation_bundle.zip` as a Kaggle Input, or run from an updated project checkout.

Add `OPENAI_API_KEY` in Kaggle Secrets. Add `GITHUB_TOKEN` with Contents read access if the release repository is private. Each new run checks GitHub Releases for the newest compatible verified adapter. Existing local run state takes precedence. Release assets carry adapter weights, so a fresh Kaggle VM starts a new evaluation and round journal from those weights.

Set `UPLOAD_ADAPTERS_TO_GITHUB = True` in the setup cell to publish completed adapters for later sessions. This requires Contents write access. The 60 general pairs use audited references with no mistake feedback. The 40 adaptive pairs use recurring errors from earlier rounds when available. Round 1 screens 40 candidates without prior feedback. The 100 development cases and final holdout stay separate from training.

The teacher API budget can pause the run. Set your total spending limit in the run cell. Preserve the entire run folder to resume progress across Kaggle sessions.


In [ ]:
from pathlib import Path
import hashlib
import os
import sys
import shutil
import zipfile

if "torch" in sys.modules and sys.modules["torch"].cuda.is_initialized():
    raise RuntimeError("Restart the Kaggle session before running setup.")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HOME"] = "/kaggle/working/hf-cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
for key in ("LOCAL_RANK", "RANK", "WORLD_SIZE", "MASTER_ADDR", "MASTER_PORT",
            "KAGGLE_ADAPTER_DATASET", "DISTILLATION_DRIVE_BACKUP"):
    os.environ.pop(key, None)

EXPECTED_BUNDLE_SHA256 = "fb3638dba104a9fb74674036acfa80726038c8af8775ae8e6e0211f72523e4ca"
EXPECTED_BUNDLE_FILES = {
    "config/distillation_loop.json": "b48ef7a06fc50eec970a09268e282a4716255b3735a4fd085da5312ba5e10fec",
    "config/self_instruct.json": "8718583f91dbfb614a0c97164f4ca480296b4c8d2773f3355a302d994d0af9b7",
    "data/evaluation/fixed_validation_v1.jsonl": "f1d10649b544ba45ba7d0d0794fd213c994bd29094de8fbcc3ed332935bf9818",
    "data/evaluation/fixed_validation_v1.manifest.json": "26ea0c0f1b350bce3243cb7b9233abe984b79cfffc7395a8e3efe0ba74a410b1",
    "data/self_instruct/GENERATION_PROMPT.md": "1da92624d98385755d9c8ddc4fc752a458dbf5ac8ae58270ebc143a12626398c",
    "data/self_instruct/seed_tasks.jsonl": "880c2abf748d4fdb4f50b71868ead6614eeaa4cd80c3cea6ca1efa7cdc5b613a",
    "data/self_instruct/task_schemas.json": "bbda704539315ee35b11287bb10baa94e0060f550c5e862073ed25a751b6b1ac",
    "distillation_loop/__init__.py": "1165e7b22f9de3dd937ff4d9bc72f815c02bf81e1658acfa067cbbd224a37d4f",
    "distillation_loop/benchmarks.py": "14161f260388dffde606b959bcc486f194191ae4691871a6e210b77568f77448",
    "distillation_loop/build_fixed_evaluation.py": "ee9d5cd27fa0a0324688590f4f575c40f8ef88f74d78933a5a5254efd458657c",
    "distillation_loop/colab.py": "a551a2d48f66b1612a4b1df7f69c8188c527d8865022dc20ca905d08a77251a0",
    "distillation_loop/controller.py": "ef7990e14dba135bfeb0eb3bbf129369fe14def987f2664e5fc0c4edc64fc2b6",
    "distillation_loop/core.py": "b9126291a2b5000380fb241cf8bf37f31d17b0abcd2b1921804648a6ac42de5e",
    "distillation_loop/correction.py": "facfeec2e82d22657d1d5e571b0b96eb083fc7ee3e9d82d2f409854b70fc494f",
    "distillation_loop/data.py": "090216009dc50a058c488ead96403e3d4e3885adc7c40ca1085d9cff1410269e",
    "distillation_loop/drive_backup.py": "abd57bd63d5625ecfa5ff11f400231b14896222aa2cb23d6d2c2f2557e03d5a7",
    "distillation_loop/dual_gpu.py": "a8c656ab5b0f50e598ae1c04cb8e3bc67ad300c54a3cfe6d27c3d1697334e94f",
    "distillation_loop/ec2.py": "1ef2f627eda1f2998c0cfb19de6c63edb9d030ce250329e5a0f2362d5d3e5294",
    "distillation_loop/evaluation.py": "221af45382c038afa88fda7895384f067d3aec618b2e9ec9cafc8d7665e4ddba",
    "distillation_loop/github_backup.py": "eb5a6d721996ff8b2fec38c35fd73a84056958a6f2f15c0480e51e9ac7c7ee2d",
    "distillation_loop/gpu_worker.py": "c5e461ae9f7b22cc33a5ab3c155ee128f1a3a72016922f413484304c49a44003",
    "distillation_loop/inference_review.py": "fe82a469438963d6c4ad656dd77153a6ccb1a7b5bd1b78354c7f38d4a2c52792",
    "distillation_loop/kaggle_backup.py": "a3e495266286c8a0319e00aea7b07194d09e5a387f043842ae3a0347f4fb4d21",
    "distillation_loop/preflight.py": "9aee8820975ef24bf7cf1784e762a34b8e3278156268042e68b26527808f6ae8",
    "distillation_loop/quality.py": "b53c0c795254d38f6ef58cf710a7055d3e45863cb79b9345e084d17552fda27a",
    "distillation_loop/review.py": "67fde59d553b7a383007261c55aecb22d4703314a91f60b43938cf8ff0e9c833",
    "distillation_loop/semantic.py": "674515731488886145f57da82efe18eff6108e97781f9c80f26c0d18c5e05428",
    "distillation_loop/smoke.py": "675947495e4e823788ea8d5063bd673556d3505f9720e3cf9be7a759b9b58438",
    "distillation_loop/student.py": "615117f99c575fb46b5a69e0579f0b1ae1c2a0192db798a1698f2ea01a7c44ad",
    "distillation_loop/trusted_templates.py": "c2cdb7decb25151c75dae1984aab5bdc32561f90c6632dbf792161fdeecb18a9",
    "requirements-colab.txt": "a1a7e53ec788d887493d663be19711ff08674edc41804f14674bfac381d0e1dd",
    "self_instruct_pipeline/__init__.py": "b1b87f51290607240ac42c881b12ae2d177948202954c7bd644dfc1b2a297924",
    "self_instruct_pipeline/budget.py": "8496b02feecaf5fe0bef47644edcd5f6bc61a620699cd437e488ea28b27ea0eb",
    "self_instruct_pipeline/config.py": "469546178a641ce6c75f8c1971e8a737ae1b54ecc206d886a9ba518a0ccbe792",
    "self_instruct_pipeline/coverage_tracker.py": "66eff73766bbf4ed9d6c776b100c3793942fdee5f2b16d14ef8b2e4ef3be0779",
    "self_instruct_pipeline/dataset_builder.py": "b90b74c12b54bf3c4129166a4d53fff2c476e2cc3e2bf01bcedf2abb67c30e93",
    "self_instruct_pipeline/deduplicator.py": "bad4629052a4cc6f8a93a2bfbed3dff24119654c811a6f335a1e28c41256659c",
    "self_instruct_pipeline/parent_backend.py": "d147d019927abbd0945e6fd48bda05e5b724c6375013519006af8b31db21fd73",
    "self_instruct_pipeline/persistence.py": "c7800cd3a48b6b88e461fccb966c7fd67845dece09067a2fbaa9900b76832255",
    "self_instruct_pipeline/pipeline.py": "4f53a39c5d325fe397b8fa1c8671cc79424f083f60004f83b364175fd7f97cf2",
    "self_instruct_pipeline/prompt_builder.py": "f780ee76ca0da0c16f1acbcb2d7ff397fc1dce794bda89aacd2804ce5e3d8b41",
    "self_instruct_pipeline/rule_checks.py": "eb43a3dbf1f463d93f34863e589b5a20f83ca43b8192b56ede1e821e907ce378",
    "self_instruct_pipeline/sampler.py": "0a0f0179d1d39dd08386045289728191b4e0de2f4501b73c344c62740b7f1e53",
    "self_instruct_pipeline/scheduler.py": "b844c1f3d3a2e36b6d700950b3cdfced331490b5cca6b362bcaaddf49c8e91e7",
    "self_instruct_pipeline/usage.py": "d9828ba5c9322fbaaaf472fd41a036ddc5ed1b2dfd5eabf8def7576e7e1427ad",
    "self_instruct_pipeline/usage_cloud.py": "cb6e1a8e4cdbf00d855904505ed951dbf242ca9082ee5d8185cd02f3e7fa4777",
    "self_instruct_pipeline/validation.py": "b8d208c63b281336aae680c8cf555cfb7f641e3d59f9e5655bb74b2aca5ed604"
}
WORKING = Path("/kaggle/working")
INPUTS = Path("/kaggle/input")
bundle_name = "colab_distillation_bundle.zip"
bundles = sorted(INPUTS.rglob(bundle_name))
if (WORKING / bundle_name).is_file():
    bundles.insert(0, WORKING / bundle_name)
BUNDLE = next((path for path in bundles
               if hashlib.sha256(path.read_bytes()).hexdigest() == EXPECTED_BUNDLE_SHA256), None)
PROJECT_ROOT = WORKING / "Health-Insurance-SLM-Colab"
if BUNDLE is not None:
    with zipfile.ZipFile(BUNDLE) as bundle:
        for entry in bundle.infolist():
            destination = (WORKING / entry.filename).resolve()
            if not destination.is_relative_to(PROJECT_ROOT.resolve()):
                raise RuntimeError("Unexpected path inside the code bundle.")
        bundle.extractall(WORKING)
elif INPUTS.is_dir():
    # Kaggle may unpack uploaded ZIPs. Verify every file before copying it.
    def matches_bundle(root):
        return all((root / name).is_file()
                   and hashlib.sha256((root / name).read_bytes()).hexdigest() == checksum
                   for name, checksum in EXPECTED_BUNDLE_FILES.items())

    extracted = next((path.parent for path in sorted(INPUTS.rglob("requirements-colab.txt"))
                      if matches_bundle(path.parent)), None)
    if extracted is not None:
        for name in EXPECTED_BUNDLE_FILES:
            destination = PROJECT_ROOT / name
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(extracted / name, destination)
    else:
        PROJECT_ROOT = Path.cwd().resolve()
else:
    PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT != WORKING / "Health-Insurance-SLM-Colab":
    if not (PROJECT_ROOT / "distillation_kaggle_100.ipynb").is_file():
        raise RuntimeError("Attach the matching code ZIP as a Kaggle Input or use the updated checkout.")
sys.path.insert(0, str(PROJECT_ROOT))
REQUIREMENTS = str(PROJECT_ROOT / "requirements-colab.txt")
print("Project:", PROJECT_ROOT)


In [ ]:
from importlib.metadata import version
import subprocess

# Preserve the CUDA-enabled PyTorch build supplied by Kaggle.
torch_constraint = Path("/kaggle/working/distillation-torch-constraint.txt")
torch_constraint.write_text(f"torch=={version('torch')}\n")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-c", str(torch_constraint),
     "-r", REQUIREMENTS],
    check=True,
)


In [ ]:
# Import Unsloth before Transformers, PEFT or TRL.
import unsloth
import torch
from kaggle_secrets import UserSecretsClient
from distillation_loop.evaluation import require_training_versions
from distillation_loop.student import require_single_gpu

require_training_versions()
require_single_gpu(torch)
print("GPU:", torch.cuda.get_device_name(0))

UPLOAD_ADAPTERS_TO_GITHUB = False
os.environ["GITHUB_ADAPTER_UPLOAD"] = "1" if UPLOAD_ADAPTERS_TO_GITHUB else "0"
secrets = UserSecretsClient()
for key in ("OPENAI_API_KEY", "GITHUB_TOKEN", "SUPABASE_URL", "SUPABASE_SECRET_KEY"):
    if not os.environ.get(key, "").strip():
        try:
            os.environ[key] = secrets.get_secret(key).strip()
        except Exception:
            pass
if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("Add OPENAI_API_KEY to Kaggle Secrets.")
if UPLOAD_ADAPTERS_TO_GITHUB and not os.environ.get("GITHUB_TOKEN"):
    raise RuntimeError("Add a Contents write GITHUB_TOKEN to enable uploads.")


In [ ]:
from distillation_loop.colab import UnslothStudentRuntime, distillation_1000_configs, save_colab_subset
from distillation_loop.core import load_loop_config
from distillation_loop.github_backup import restore_latest_release
from distillation_loop.preflight import readiness
from self_instruct_pipeline.config import load_config
from self_instruct_pipeline.persistence import atomic_json
import json

RUN_DIR = WORKING / "distillation_kaggle_1000_unsloth_v1"
RUN_DIR.mkdir(parents=True, exist_ok=True)
MODEL_NAME = "Qwen/Qwen3-8B"
parent_config, loop_config = distillation_1000_configs(
    load_config(PROJECT_ROOT / "config/self_instruct.json"),
    load_loop_config(str(PROJECT_ROOT / "config/distillation_loop.json")),
    RUN_DIR,
)

seed = restore_latest_release(RUN_DIR, MODEL_NAME)
if seed:
    loop_config["training"].update(seed["training"])
    print("Pinned GitHub release:", seed["tag"], seed["sha256"])
elif (RUN_DIR / "loop_state.json").exists():
    print("Resuming existing local run.")
else:
    print("No compatible GitHub adapter release found. Starting from the base model.")

profile_path = RUN_DIR / "requested_loop_config.json"
atomic_json(profile_path, loop_config)
loop_config = load_loop_config(str(profile_path))
report = readiness(parent_config, loop_config, check_runtime=False)
if not report["ready"]:
    raise RuntimeError(f"Resolve the data/configuration issues: {report}")
schemas = json.loads((PROJECT_ROOT / "data/self_instruct/task_schemas.json").read_text())["tasks"]
subset = save_colab_subset(loop_config, schemas, RUN_DIR)
print("Evaluation cases:", subset["total"], "Round cap:", loop_config["max_rounds"])
student_runtime = UnslothStudentRuntime.load(MODEL_NAME, loop_config)


In [ ]:
from distillation_loop.controller import run_loop
from self_instruct_pipeline.budget import APIBudget
from self_instruct_pipeline.usage_cloud import SupabaseLedger, sync_usage_history

ledger = SupabaseLedger.from_environment()
if ledger:
    ledger.check()
    print(sync_usage_history(RUN_DIR))
# To resume a budget-stopped run, raise the exhausted cap and rerun this cell.
# Limits are cumulative for RUN_DIR. Existing usage and training state are retained.
# Set a total spending cap you are comfortable with before starting.
API_MAX_CALLS = 10000
API_MAX_COST_USD = "2.00"
budget = APIBudget(RUN_DIR / "api_budget", max_calls=API_MAX_CALLS, max_cost_usd=API_MAX_COST_USD)
with budget.activate(allow_increase=True):
    result = run_loop(student_runtime, parent_config, loop_config,
                      initial_adapter=seed["checkpoint"] if seed else None)

print("Completed rounds:", result["completed_rounds"])
print("Stop reason:", result["status"])
print("Selected checkpoint:", result["best_checkpoint"])
